# Human-Condition Clustered vs. Naive SE Check

Extends the Baseline clustered/naive SE ratio check already reported in
`Exp5_RS_Persistence_Power_Analysis.md` (ratio 0.94x-1.02x, "blocks within a session
behave close to independent") to the **Human** condition, in both exp4 and
exp5-prescreen's frozen pilot data.

**Question:** does treating blocks as independent (naive SE) understate the true
uncertainty of the block-weighted mean paired-delta H_RS, once blocks are correctly
resampled by whole session (clustered SE)? A ratio near 1 means no, validating the
sigma_within=0.064 pooled figure used in the Exp5 power analysis for Human data the
same way it was validated for Baseline. A ratio meaningfully above 1 would mean real
within-session drift (e.g. fatigue/engagement fading across an 80-block session) is
present and the power model's sigma_within is understated for Human specifically.

### Load frozen data

Expects the four frozen files below either in this notebook's working directory or
its `data/` subfolder (matches the search convention already used in the NB1/NB2
Colab notebooks). No other setup required.

In [ ]:
from pathlib import Path
import ast
import numpy as np
import pandas as pd

REQUIRED_INPUTS = {
    "exp4_blocks": "Frozen_Blocks_2026-02-10_195735.csv",
    "exp4_sessions": "Frozen_Sessions_2026-02-10_195735.csv",
    "exp5p_blocks": "Frozen_Exp5Prescreen_Blocks_2026-07-25.csv",
    "exp5p_sessions": "Frozen_Exp5Prescreen_Sessions_2026-07-25.csv",
}

def locate_required_inputs(filenames, search_root=None):
    root = Path.cwd() if search_root is None else Path(search_root)
    resolved, problems = {}, []
    for label, filename in filenames.items():
        candidates = []
        for parent in (root, root / "data"):
            if parent.is_dir():
                candidates.extend(p.resolve() for p in parent.glob(filename) if p.is_file())
        if not candidates:
            candidates = [p.resolve() for p in root.rglob(filename) if p.is_file()]
        candidates = sorted(set(candidates))
        if len(candidates) == 1:
            resolved[label] = str(candidates[0])
        elif len(candidates) == 0:
            problems.append(f"MISSING: {filename}")
        else:
            problems.append(f"DUPLICATED input for {label}: {candidates}")
    if problems:
        raise FileNotFoundError(
            "Frozen input-file check failed.\n" + "\n".join(f"  - {p}" for p in problems) +
            f"\nSearch root: {root.resolve()}"
        )
    return resolved

INPUT_PATHS = locate_required_inputs(REQUIRED_INPUTS)
print("Resolved inputs:")
for label, path in INPUT_PATHS.items():
    print(f"  {label}: {path}")

### Shared functions — R/S ordering score, clustered-vs-naive SE

In [ ]:
rng = np.random.default_rng(20260822)
N_BOOT = 10_000

def hurst_approx(x_bits_2d, n_bits):
    """Single-scale R/S ordering score, vectorized."""
    x = x_bits_2d.astype(np.float64) * 2 - 1
    mean_ = x.mean(axis=1, keepdims=True)
    d = x - mean_
    y = np.cumsum(d, axis=1)
    minY = np.minimum(0, y.min(axis=1))
    maxY = np.maximum(0, y.max(axis=1))
    R = maxY - minY
    S = np.sqrt((d * d).mean(axis=1))
    S[S == 0] = 1
    ratio = np.where(R / S == 0, 1, R / S)
    h = np.log(ratio) / np.log(n_bits)
    return np.clip(h, 0, 1)

def clustered_vs_naive_se(session_ids, delta_h, n_boot=N_BOOT, seed_rng=rng):
    """Naive block-level SE vs. session-clustered bootstrap SE, both on the
    block-weighted mean of delta_h. Mirrors the Baseline computation already
    reported in Exp5_RS_Persistence_Power_Analysis.md."""
    n_blocks = len(delta_h)
    naive_se = delta_h.std(ddof=1) / np.sqrt(n_blocks)

    df = pd.DataFrame({"session": session_ids, "dh": delta_h})
    grouped = df.groupby("session")["dh"].agg(["sum", "count"])
    keys = grouped.index.to_numpy()
    n_sessions = len(keys)
    boot_means = np.empty(n_boot)
    for i in range(n_boot):
        sampled = seed_rng.choice(keys, size=n_sessions, replace=True)
        s = grouped.loc[sampled, "sum"].sum()
        c = grouped.loc[sampled, "count"].sum()
        boot_means[i] = s / c
    clustered_se = boot_means.std(ddof=1)

    return {
        "n_blocks": n_blocks, "n_sessions": n_sessions,
        "naive_se": naive_se, "clustered_se": clustered_se,
        "ratio": clustered_se / naive_se,
    }

### exp4 Human

In [ ]:
blocks4 = pd.read_csv(INPUT_PATHS["exp4_blocks"])
sess4 = pd.read_csv(INPUT_PATHS["exp4_sessions"])

def parse_td(v):
    try:
        p = ast.literal_eval(v)
        return p if isinstance(p, dict) else None
    except Exception:
        return None

parsed = blocks4["trial_data"].apply(parse_td)
blocks4["subject_bits"] = parsed.apply(lambda p: p.get("subject_bits") if p else None)
blocks4["demon_bits"] = parsed.apply(lambda p: p.get("demon_bits") if p else None)
blocks4 = blocks4[blocks4["subject_bits"].notna() & blocks4["demon_bits"].notna()].copy()

subj4 = np.array(blocks4["subject_bits"].tolist(), dtype=np.int8)
dem4 = np.array(blocks4["demon_bits"].tolist(), dtype=np.int8)
blocks4["delta_h"] = hurst_approx(subj4, subj4.shape[1]) - hurst_approx(dem4, dem4.shape[1])

sess4r = sess4.rename(columns={"sessionId": "session_id", "agent_class": "condition"})
blocks4 = blocks4.rename(columns={"sessionId": "session_id"}).merge(
    sess4r[["session_id", "condition"]], on="session_id", how="left"
)

h4 = blocks4[blocks4.condition == "human"]
result4 = clustered_vs_naive_se(h4.session_id.to_numpy(), h4.delta_h.to_numpy())
print(f"exp4 Human:  blocks={result4['n_blocks']}  sessions={result4['n_sessions']}  "
      f"naive_SE={result4['naive_se']:.6f}  clustered_SE={result4['clustered_se']:.6f}  "
      f"ratio={result4['ratio']:.3f}x")

### exp5-prescreen Human

In [ ]:
blocks5 = pd.read_csv(INPUT_PATHS["exp5p_blocks"])
sess5 = pd.read_csv(INPUT_PATHS["exp5p_sessions"])

def to_bits(s):
    if isinstance(s, str):
        s2 = s.strip("[]").replace(",", " ")
        return np.array([int(x) for x in s2.split()], dtype=np.int8)
    return None

blocks5["subject_arr"] = blocks5["subject_bits"].apply(to_bits)
blocks5["demon_arr"] = blocks5["demon_bits"].apply(to_bits)
blocks5 = blocks5[
    blocks5["subject_arr"].apply(lambda a: a is not None)
    & blocks5["demon_arr"].apply(lambda a: a is not None)
].copy()

# Keep only the modal bit-length (drops any malformed/truncated rows).
lens = blocks5["subject_arr"].apply(len)
blocks5 = blocks5[lens == lens.mode()[0]].copy()

subj5 = np.stack(blocks5["subject_arr"].to_numpy())
dem5 = np.stack(blocks5["demon_arr"].to_numpy())
blocks5["delta_h"] = hurst_approx(subj5, subj5.shape[1]) - hurst_approx(dem5, dem5.shape[1])
blocks5 = blocks5.merge(sess5[["session_id", "condition"]], on="session_id", how="left")

h5 = blocks5[blocks5.condition == "human"]
result5 = clustered_vs_naive_se(h5.session_id.to_numpy(), h5.delta_h.to_numpy())
print(f"exp5-prescreen Human:  blocks={result5['n_blocks']}  sessions={result5['n_sessions']}  "
      f"naive_SE={result5['naive_se']:.6f}  clustered_SE={result5['clustered_se']:.6f}  "
      f"ratio={result5['ratio']:.3f}x")

print()
print("Reference -- Baseline ratios already reported in Exp5_RS_Persistence_Power_Analysis.md:")
print("  exp4 Baseline: 0.94x   exp5-prescreen Baseline: 1.02x")

### Interpretation boundary

A ratio near 1 (in either direction) means blocks within a session behave close to
independent for the block-weighted mean of paired-delta H_RS -- the same conclusion
already reached for Baseline. A ratio meaningfully *above* 1 would be the concerning
direction: it would mean real within-session drift is inflating true uncertainty
beyond what the naive block-level SE assumes, and the sigma_within=0.064 figure (and
therefore the N=120/200 power figures) would need the same decomposition correction
already applied to Baseline's between-session SD.

A ratio *below* 1 is not evidence of that problem -- if anything it means the naive
SE was already conservative relative to the clustered estimate for that dataset.
With fewer sessions (e.g. exp5-prescreen's 35 Human sessions vs. exp4's 159), this
ratio is a noisier estimate and shouldn't be over-read as a precise number, only as
a directional check.